In [1]:
import sys, os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_23_try_0.pickle

In [3]:
%%RecordEvent
%%time
### cell 24 ###

# Optimize filtering and grouping for speed and memory
use_modin = os.environ.get("IREWR_WITH_MODIN") == "True"

# 1) Fast boolean filter using .isin
mask_countries = df['first_country'].isin(['USA', 'India'])
us_ind = df.loc[mask_countries]

# 2) If running under Modin, convert to pandas once
if use_modin:
    df = df._to_pandas()

# 3) Re-compute mask on the (possibly converted) DataFrame
mask_group = df['first_country'].isin(['USA', 'India'])

# 4) Group on the reduced frame; use .size + unstack(fill_value=0) instead of value_counts+fillna
#    cast to float to match original fillna behavior
data_sub = (
    df.loc[mask_group, ['first_country', 'year_added']]
      .groupby(['first_country', 'year_added'])
      .size()
      .unstack(fill_value=0)
      .astype(float)
      .loc[['USA', 'India']]
      .cumsum(axis=0)
      .T
)

# 5) If Modin, wrap outputs as pandas.DataFrame to match original
if use_modin:
    df = pd.DataFrame(df)
    data_sub = pd.DataFrame(data_sub)

CPU times: user 18 ms, sys: 150 µs, total: 18.1 ms
Wall time: 18.1 ms


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_24_try_0.pickle